In [2]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import optuna
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.metrics import root_mean_squared_error
from xgboost import XGBRegressor
from Preprocess import preprocess_data_window
from catboost import CatBoostRegressor

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [3]:
# ── Cell 2: Load data ────────────────────────────────────────────────────────
data_orig     = pd.read_csv("../Whillians-GPS-Data-and-Features.csv")
filtered_time = pd.read_csv("../filtered_time_to_next_event.csv")

In [ ]:
# ── Cell 3: Preprocessing ────────────────────────────────────────────────────
n_previous_events = 20
n_qubits_base = 6  # features per event

X_train, X_val, X_test, y_train, y_val, y_test, feature_cols = preprocess_data_window(
    filtered_time, data_orig, n_previous_events
)

print("NaN count per column:\n", X_train.isna().sum())
print("Total NaN:", X_train.isna().sum().sum())

import pandas as pd
print(pd.Series(y_train).describe())
print("\nSkew:", pd.Series(y_train).skew())

X shape:  (2822, 126)
y shape:  (2822,)
NaN count per column:
 tide_deriv-0      0
form_fac-0        0
time_since-0      0
slip_size-0       0
high_t_evt-0      0
                 ..
form_fac-20       0
time_since-20     0
slip_size-20      0
high_t_evt-20     0
tide_height-20    0
Length: 126, dtype: int64
Total NaN: 0
count      1692.000000
mean      60206.888298
std       20347.145025
min       23265.000000
25%       43316.250000
50%       54832.500000
75%       82443.750000
max      119220.000000
Name: time_to_next_ev_hr, dtype: float64

Skew: 0.27697104661890737


In [5]:
# ── Cell 4: Config ───────────────────────────────────────────────────────────
QRC_CONFIG = {
    "num_layers_per_event": 2,
    "shots": 4096,
    "n_iterations": 1,
    "top_k": 3,
    "correlation_threshold": 0.0,
    "random_seed": 42,
}

CLASSICAL_MODEL_NAME = "XGBoost"
n_previous_events = 20

print("QRC Config:", QRC_CONFIG)
print("Classical model:", CLASSICAL_MODEL_NAME)

QRC Config: {'num_layers_per_event': 2, 'shots': 4096, 'n_iterations': 1, 'top_k': 3, 'correlation_threshold': 0.0, 'random_seed': 42}
Classical model: XGBoost


In [6]:
# ── Cell 5+6: Scaling only — no compression ──────────────────────────────────
def scale_to_pi_range(X_train, X_val, X_test):
    train_min = X_train.min(axis=0)
    train_max = X_train.max(axis=0)
    denom = train_max - train_min
    denom[denom == 0] = 1.0

    def transform(X):
        scaled = (X - train_min) / denom
        scaled = np.clip(scaled, 0.0, 1.0)
        return scaled * np.pi

    return transform(X_train), transform(X_val), transform(X_test)


X_train_sel = X_train.to_numpy()
X_val_sel   = X_val.to_numpy()
X_test_sel  = X_test.to_numpy()

X_train_q, X_val_q, X_test_q = scale_to_pi_range(X_train_sel, X_val_sel, X_test_sel)

# Group 21 events into blocks of n_qubits_base features each
# Each block gets its own circuit
n_total_events = n_previous_events + 1   # 21
n_circuits     = n_total_events           # one circuit per event = 21 circuits
n_qubits       = n_qubits_base            # 6 qubits per circuit
n_states       = 2 ** n_qubits            # 64

print(f"Total events: {n_total_events}")
print(f"Circuits to run per sample: {n_circuits}")
print(f"Qubits per circuit: {n_qubits}")
print(f"Prob vector per circuit: {n_states}")
print(f"Total readout dim (quantum): {n_circuits * n_states}")
print(f"Total readout dim (hybrid): {n_circuits * n_states + X_train_q.shape[1]}")

Total events: 21
Circuits to run per sample: 21
Qubits per circuit: 6
Prob vector per circuit: 64
Total readout dim (quantum): 1344
Total readout dim (hybrid): 1470


In [7]:
# ── Cell 7: Circuit primitives ───────────────────────────────────────────────
def generate_random_angles(num_layers, n_qubits, rng):
    return rng.uniform(0, 2 * np.pi, size=(num_layers, n_qubits, 3))


def encode_rotations(qc, data_sample, n_qubits):
    for i in range(n_qubits):
        qc.ry(float(data_sample[i]), i)


def build_reservoir_circuit_rotations(data_sample, random_angles, num_layers, n_qubits):
    qc = QuantumCircuit(n_qubits)

    encode_rotations(qc, data_sample, n_qubits)
    qc.barrier()

    for layer in range(num_layers):
        for i in range(n_qubits):
            qc.rx(float(random_angles[layer, i, 0]), i)
            qc.rz(float(random_angles[layer, i, 1]), i)
            qc.ry(float(random_angles[layer, i, 2]), i)
        for i in range(n_qubits - 1):
            qc.cx(i, i + 1)
        qc.cx(n_qubits - 1, 0)
        qc.barrier()

        encode_rotations(qc, data_sample, n_qubits)
        qc.barrier()

        for i in range(n_qubits):
            qc.rx(float(random_angles[layer, i, 2]), i)
            qc.rz(float(random_angles[layer, i, 0]), i)
            qc.ry(float(random_angles[layer, i, 1]), i)
        for i in range(n_qubits - 1, 0, -1):
            qc.cx(i, i - 1)
        qc.cx(0, n_qubits - 1)
        qc.barrier()

    qc.measure_all()
    return qc

In [8]:
# ── Cell 8: Multi-circuit reservoir runner ────────────────────────────────────
def run_quantum_reservoir_multi(
    X_data, angle_bank, num_layers, n_qubits, n_total_events, shots, batch_size=512
):
    """
    Run one independent 6-qubit circuit per event slot, for every sample.

    X_data     : (m, n_total_events * n_qubits) — full windowed feature matrix
    angle_bank : list of n_total_events arrays, each (num_layers, n_qubits, 3)
                 Each event slot gets its own random reservoir weights.

    Returns
    -------
    prob_matrix : (m, n_total_events * 2**n_qubits)
        All per-event probability vectors concatenated horizontally.
    """
    if hasattr(X_data, 'values'):
        X_data = X_data.to_numpy()

    m        = X_data.shape[0]
    n_states = 2 ** n_qubits
    simulator = AerSimulator()

    # Output: one 64-dim vector per event per sample
    prob_matrix = np.zeros((m, n_total_events * n_states))

    for event_idx in range(n_total_events):
        # Extract this event's 6 features for all samples
        start = event_idx * n_qubits
        end   = start + n_qubits
        X_event = X_data[:, start:end]   # (m, 6)

        random_angles = angle_bank[event_idx]  # (num_layers, n_qubits, 3)

        # Build circuits for all samples for this event
        circuits = [
            build_reservoir_circuit_rotations(X_event[i], random_angles, num_layers, n_qubits)
            for i in range(m)
        ]

        # Execute in batches
        for batch_start in range(0, m, batch_size):
            batch_end = min(batch_start + batch_size, m)
            batch     = circuits[batch_start:batch_end]
            result    = simulator.run(batch, shots=shots).result()
            counts_list = result.get_counts()
            if isinstance(counts_list, dict):
                counts_list = [counts_list]

            for j, counts in enumerate(counts_list):
                for bitstring, count in counts.items():
                    idx = int(bitstring.replace(" ", ""), 2)
                    out_col = event_idx * n_states + idx
                    prob_matrix[batch_start + j, out_col] = count / shots

        print(f"  Event {event_idx + 1}/{n_total_events} done", end="\r")

    print()
    return prob_matrix


def make_hybrid_features(P, X_classical):
    """Concatenate quantum prob vectors with all raw classical features."""
    return np.hstack([P, X_classical])

In [9]:
# ── Cell 9: Model registry ───────────────────────────────────────────────────
def get_model_registry():
    return {
        "Ridge": {
            "optuna_fn": lambda trial: {
                "alpha": trial.suggest_float("alpha", 1e-2, 1e8, log=True),
            },
            "fixed_params": {},
            "build_fn": lambda params: Ridge(**params),
            "fit_fn": lambda model, Xt, yt, Xv, yv: model.fit(Xt, yt),
            "shap_explainer": "linear",
        },
        "XGBoost": {
            "optuna_fn": lambda trial: {
                "learning_rate":    trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
                "max_depth":        trial.suggest_int("max_depth", 2, 6),
                "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.8),
                "min_child_weight": trial.suggest_int("min_child_weight", 3, 20),
                # Push harder on regularization — 1470-dim input needs it
                "reg_alpha":        trial.suggest_float("reg_alpha", 0.1, 50.0, log=True),
                "reg_lambda":       trial.suggest_float("reg_lambda", 0.1, 50.0, log=True),
                # Limit tree complexity on high-dim noisy input
                "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
                "max_delta_step":   trial.suggest_int("max_delta_step", 0, 5),
            },
            "fixed_params": {
                "objective":             "reg:squarederror",
                "n_estimators":          2000,
                "random_state":          42,
                "early_stopping_rounds": 75,  # more patience on larger feature set
                "tree_method":           "hist",  # faster on wide feature matrices
            },
            "build_fn": lambda params: XGBRegressor(**params),
            "fit_fn": lambda model, Xt, yt, Xv, yv: model.fit(
                Xt, yt, eval_set=[(Xv, yv)], verbose=False
            ),
            "shap_explainer": "tree",
        },
        "CatBoost": {
            "optuna_fn": lambda trial: {
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "depth":         trial.suggest_int("depth", 2, 8),
            },
            "fixed_params": {
                "iterations":          100,
                "loss_function":       "RMSE",
                "eval_metric":         "RMSE",
                "random_seed":         42,
                "verbose":             False,
                "allow_writing_files": False,
            },
            "build_fn": lambda params: CatBoostRegressor(**params),
            "fit_fn": lambda model, Xt, yt, Xv, yv: model.fit(
                Xt, yt, eval_set=(Xv, yv), use_best_model=True
            ),
            "shap_explainer": "tree",
        },
        "RandomForest": {
            "optuna_fn": lambda trial: {
                "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
                "max_depth":    trial.suggest_int("max_depth", 2, 10),
            },
            "fixed_params": {"random_state": 42},
            "build_fn": lambda params: RandomForestRegressor(**params),
            "fit_fn": lambda model, Xt, yt, Xv, yv: model.fit(Xt, yt),
            "shap_explainer": "tree",
        },
    }


registry = get_model_registry()
print(f"Available models: {list(registry.keys())}")
print(f"Using: {CLASSICAL_MODEL_NAME}")

Available models: ['Ridge', 'XGBoost', 'CatBoost', 'RandomForest']
Using: XGBoost


In [10]:
# ── Cell 10: Single iteration — uses multi-circuit reservoir ──────────────────
def run_single_qrc_iteration(
    iteration_idx,
    X_train_q, X_val_q, X_test_q,
    y_train, y_val, y_test,
    n_qubits, config, model_name, registry,
):
    num_layers    = config["num_layers_per_event"]
    shots         = config["shots"]
    seed          = config["random_seed"] + iteration_idx
    n_total_events = n_previous_events + 1  # 21

    rng = np.random.default_rng(seed)

    # One independent angle set per event slot
    angle_bank = [
        generate_random_angles(num_layers, n_qubits, rng)
        for _ in range(n_total_events)
    ]

    print(f"  Running reservoir on train ({X_train_q.shape[0]} samples)...")
    P_train = run_quantum_reservoir_multi(
        X_train_q, angle_bank, num_layers, n_qubits, n_total_events, shots
    )
    print(f"  Running reservoir on val ({X_val_q.shape[0]} samples)...")
    P_val = run_quantum_reservoir_multi(
        X_val_q, angle_bank, num_layers, n_qubits, n_total_events, shots
    )
    print(f"  Running reservoir on test ({X_test_q.shape[0]} samples)...")
    P_test = run_quantum_reservoir_multi(
        X_test_q, angle_bank, num_layers, n_qubits, n_total_events, shots
    )

    # Hybrid: all quantum prob vectors + all 126 raw features
    H_train = make_hybrid_features(P_train, X_train_q)
    H_val   = make_hybrid_features(P_val,   X_val_q)
    H_test  = make_hybrid_features(P_test,  X_test_q)

    entry = registry[model_name]

    def objective(trial):
        tuned  = entry["optuna_fn"](trial)
        params = {**entry["fixed_params"], **tuned}
        model  = entry["build_fn"](params)
        entry["fit_fn"](model, H_train, y_train, H_val, y_val)
        preds  = model.predict(H_val)
        return mean_absolute_error(y_val, preds)

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=seed),
    )
    study.optimize(objective, n_trials=50)

    best_tuned  = entry["optuna_fn"](study.best_trial)
    best_params = {**entry["fixed_params"], **best_tuned}
    model       = entry["build_fn"](best_params)
    entry["fit_fn"](model, H_train, y_train, H_val, y_val)

    val_pred  = model.predict(H_val)
    test_pred = model.predict(H_test)

    return {
        "iteration":   iteration_idx,
        "val_r2":      r2_score(y_val, val_pred),
        "val_rmse":    root_mean_squared_error(y_val, val_pred),
        "val_mae":     mean_absolute_error(y_val, val_pred),
        "test_rmse":   root_mean_squared_error(y_test, test_pred),
        "test_mae":    mean_absolute_error(y_test, test_pred),
        "test_pred":   test_pred,
        "best_params": best_params,
        "model":       model,
        "P_train": P_train, "P_val": P_val, "P_test": P_test,
    }

In [11]:
# ── Cell 11: Pipeline + run ──────────────────────────────────────────────────
def run_qrc_pipeline(
    X_train_q, X_val_q, X_test_q,
    y_train, y_val, y_test,
    n_qubits, config, model_name, registry,
):
    results = []
    for i in range(config["n_iterations"]):
        print(f"\nIteration {i + 1}/{config['n_iterations']}:")
        res = run_single_qrc_iteration(
            i,
            X_train_q, X_val_q, X_test_q,
            y_train, y_val, y_test,
            n_qubits, config, model_name, registry,
        )
        print(
            f"\tVal R2: {res['val_r2']:.4f} | Val RMSE: {res['val_rmse']:.2f} | "
            f"Val MAE: {res['val_mae']:.2f} | Test RMSE: {res['test_rmse']:.2f} | "
            f"Test MAE: {res['test_mae']:.2f}"
        )
        results.append(res)

    top_k          = config["top_k"]
    sorted_results = sorted(results, key=lambda r: r["val_mae"])
    top_results    = sorted_results[:top_k]
    top_indices    = [r["iteration"] for r in top_results]

    print(f"\nTop-{top_k} iterations (by val MAE): {[i + 1 for i in top_indices]}")

    ensemble_pred = np.mean([r["test_pred"] for r in top_results], axis=0)
    ensemble_rmse = root_mean_squared_error(y_test, ensemble_pred)
    ensemble_mae  = mean_absolute_error(y_test, ensemble_pred)
    ensemble_r2   = r2_score(y_test, ensemble_pred)

    print(f"\nEnsemble Test RMSE: {ensemble_rmse:.2f}")
    print(f"Ensemble Test MAE:  {ensemble_mae:.2f}")
    print(f"Ensemble Test R2:   {ensemble_r2:.4f}")

    return results, ensemble_pred, top_indices


all_results, ensemble_pred, top_indices = run_qrc_pipeline(
    X_train_q,
    X_val_q,
    X_test_q,
    y_train,
    y_val,
    y_test,
    n_qubits,
    QRC_CONFIG,
    "XGBoost",
    registry,
)


Iteration 1/1:
  Running reservoir on train (1692 samples)...
  Event 21/21 done
  Running reservoir on val (565 samples)...
  Event 21/21 done
  Running reservoir on test (565 samples)...
  Event 21/21 done
	Val R2: 0.2665 | Val RMSE: 17183.84 | Val MAE: 13383.39 | Test RMSE: 15454.76 | Test MAE: 11982.24

Top-3 iterations (by val MAE): [1]

Ensemble Test RMSE: 15454.76
Ensemble Test MAE:  11982.24
Ensemble Test R2:   0.4112


In [12]:
# ── Cell 12: Classical baseline diagnostic ───────────────────────────────────
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

last_res = all_results[-1]
P_dim = last_res["P_train"].shape[1]

H_train = np.hstack([last_res["P_train"], X_train_q])
H_val   = np.hstack([last_res["P_val"],   X_val_q])
H_test  = np.hstack([last_res["P_test"],  X_test_q])

X_tr_raw = H_train[:, P_dim:]
X_v_raw  = H_val[:,   P_dim:]
X_te_raw = H_test[:,  P_dim:]

classical_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=1.0,
    early_stopping_rounds=75,
    tree_method="hist",
    random_state=42,
)

classical_model.fit(
    X_tr_raw, y_train,
    eval_set=[(X_v_raw, y_val)],
    verbose=False
)

classical_pred = classical_model.predict(X_te_raw)
classical_mae  = mean_absolute_error(y_test, classical_pred)
quantum_mae    = last_res["test_mae"]

print(f"Classical XGBoost (126 features, no quantum): MAE = {classical_mae:.2f}")
print(f"Quantum hybrid (single iteration):            MAE = {quantum_mae:.2f}")
print(f"Quantum vs classical difference:              {quantum_mae - classical_mae:.2f}s")

Classical XGBoost (126 features, no quantum): MAE = 11671.34
Quantum hybrid (single iteration):            MAE = 11982.24
Quantum vs classical difference:              310.90s
